<a href="https://colab.research.google.com/github/elbuh0/genai-course/blob/main/genai-course/week6/homework/Week_6_HW_Prompt_Engineering_Audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Homework 6 — Prompt Engineering Audit
COM SCI-X 455 · Week 6 · Megan Armani, PhD

#Setup

In [2]:
%pip install -q openai pydantic pandas tabulate tiktoken

import os, json, re
from typing import List, Literal, Optional
import pandas as pd
import tiktoken
from pydantic import BaseModel, Field, ValidationError
from openai import OpenAI
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("openai")
client = OpenAI()
MODEL = "gpt-4o-mini"

try:
    enc = tiktoken.encoding_for_model(MODEL)
except KeyError:
    enc = tiktoken.get_encoding("o200k_base")

def count_tokens(s):
    return len(enc.encode(s or ""))

print(f"Ready. Model={MODEL}, tokenizer={enc.name}")

Ready. Model=gpt-4o-mini, tokenizer=o200k_base


# Part 1: Tasks with Prompt Variations

In [15]:
# Task: Researcher - Recommending and/or evaluating methodology gaps

import json
from typing import List, Literal, Optional
from pydantic import BaseModel, Field, ValidationError, model_validator

# Pydantic model used to validate JSON outputs (V4, V5)
class AreaAssessment(BaseModel):
    status: Literal["adequate", "gap", "not_stated", "not_applicable"]
    gap: Optional[str] = None
    fix: Optional[str] = None
    severity: Optional[Literal["high", "medium", "low"]] = None

    @model_validator(mode="after")
    def _needs_fix(self):
        if self.status in ("gap", "not_stated") and not (self.gap and self.fix and self.severity):
            raise ValueError("status gap/not_stated requires gap, fix and severity")
        return self

class Assessment(BaseModel):
    mode: Literal["recommend", "gap_analysis"]
    research_question: str
    data_and_annotation: AreaAssessment
    split_and_leakage: AreaAssessment
    benchmarking_and_baselines: AreaAssessment
    controlled_experiments: AreaAssessment
    metrics_and_significance: AreaAssessment
    generalisation_and_domain_shift: AreaAssessment
    recommended_methodology: str

SCHEMA_JSON = json.dumps(Assessment.model_json_schema())

# Six required areas (used explicitly by V3–V5)
AREAS = """Assess the methodology on these six areas:
1. Data and Annotation (source, size, class balance, representativeness, label quality)
2. Training/Validation/Test Split (separation, leakage)
3. Benchmarking and Baselines (comparison against baselines / state of the art)
4. Controlled Experiments and Ablations (one variable at a time, confounds)
5. Metrics and Statistical Significance (right metrics, confidence intervals, repeated runs)
6. Generalisation and Domain Shift (real-world/unseen conditions, synthetic data realism)
"""

ROLE_AND_JSON = """You are a Senior R&D Engineer at a military technology startup.
You mentor junior employees with little experience. There are minimal resources allotted
to these individuals. Your job is to assess the methodology of a proposal and recommend
80/20 solutions using free open-source solutions that answer a basic research question
that can be pursued as a proof-of-concept by a novice employee.

""" + AREAS + """
Return ONLY a JSON object in exactly this structure, with no other text:

{
  "mode": "recommend" | "gap_analysis",
  "research_question": "string",
  "data_and_annotation": {"status": "adequate" | "gap" | "not_stated" | "not_applicable", "gap": "string" | null, "fix": "string" | null, "severity": "high" | "medium" | "low" | null},
  "split_and_leakage": {same fields},
  "benchmarking_and_baselines": {same fields},
  "controlled_experiments": {same fields},
  "metrics_and_significance": {same fields},
  "generalisation_and_domain_shift": {same fields},
  "recommended_methodology": "string"
}

Rules:
- "recommend" if the text states no methodology; "gap_analysis" if any approach is stated.
- status: "adequate" = handled well; "gap" = stated but flawed; "not_stated" = missing;
  "not_applicable" = irrelevant to this study.
- If status is "gap" or "not_stated", give gap, fix and severity; otherwise set them to null.
- In "recommend" mode, every area is "not_stated" and each fix is your recommendation for that area.
"""

# V2 few-shot prompt
V2_USER = """Task: You will assess the methodology of a research proposal.

Example 1:
Input: We fine-tuned a ResNet-50 on 1,500 smartphone photos of crop leaves (healthy vs. diseased),
collected on one farm and labelled by the author, split randomly 80/20 by image. Accuracy was 97%.
Output:
MODE: gap_analysis
RESEARCH QUESTION: Can a fine-tuned CNN detect leaf disease from smartphone photos?
1. DATA AND ANNOTATION: [GAP | HIGH] Single farm and a single unverified labeller. Fix: add a second site; have a second annotator label a subset and report agreement (Cohen's kappa).
2. TRAIN/VAL/TEST SPLIT: [GAP | HIGH] Random split by image puts photos of the same plant in both sets, so the network can memorise individual plants (leaf shape, background, lighting) rather than learn the disease, which inflates the 97%. Fix: split by plant or by field, so test plants are never seen in training.
3. BENCHMARKING AND BASELINES: [NOT STATED | MEDIUM] No comparison. Fix: compare with a simple baseline (logistic regression on colour features) and an off-the-shelf model.
4. CONTROLLED EXPERIMENTS: [NOT STATED | LOW] No ablations. Fix: test with and without augmentation and pre-training.
5. METRICS AND SIGNIFICANCE: [GAP | MEDIUM] Accuracy only, single run. Fix: report precision/recall per class with 95% CIs over 3 seeds.
6. GENERALISATION AND DOMAIN SHIFT: [GAP | MEDIUM] One farm, one phone. Fix: hold out a second farm as an external test set.
RECOMMENDED METHODOLOGY: Collect from two farms, split by plant, double-label a subset, compare against two baselines, report per-class metrics with CIs over 3 seeds, test on the held-out farm.

Example 2:
Input: Urban delivery robots often get stuck at curbs. We want to know whether a cheap depth camera can reliably detect curbs at night.
Output:
MODE: recommend
RESEARCH QUESTION: Can a low-cost depth camera reliably detect curbs at night?
1. DATA AND ANNOTATION: [NOT STATED | HIGH] Fix: record about 2 hours of night footage across at least 3 neighbourhoods; label curb and no-curb frames with two annotators.
2. TRAIN/VAL/TEST SPLIT: [NOT STATED | HIGH] Fix: split by route, so no street appears in both training and test.
3. BENCHMARKING AND BASELINES: [NOT STATED | MEDIUM] Fix: compare with a daytime-trained model and a simple height-threshold rule.
4. CONTROLLED EXPERIMENTS: [NOT STATED | MEDIUM] Fix: vary one factor at a time (lighting, wet vs. dry, speed).
5. METRICS AND SIGNIFICANCE: [NOT STATED | MEDIUM] Fix: report recall on curbs (missing one is costly) with bootstrap CIs.
6. GENERALISATION AND DOMAIN SHIFT: [NOT STATED | MEDIUM] Fix: test in a neighbourhood not used for training.
RECOMMENDED METHODOLOGY: Collect night footage on several routes, split by route, compare against two baselines, run one-factor-at-a-time trials, and report curb recall with CIs on an unseen neighbourhood.

Input: {text}
Output:
"""

# Prompts
PROMPTS = {
    # V1: minimal instruction, no examples
    "V1_zero_shot": {"label": "V1 Zero-shot", "format": "text",
        "system": "",
        "user": """Task: You will assess the methodology of a research proposal.
Start your answer with MODE: recommend or MODE: gap_analysis.

Proposal:
{text}
"""},

    # V2: minimal instruction; the examples demonstrate the expected output
    "V2_few_shot": {"label": "V2 Few-shot", "format": "text",
        "system": "",
        "user": V2_USER},

    "V3_cot": {"label": "V3 Chain-of-thought", "format": "text",
        "system": "",
        "user": """Task: Assess the methodology of a research proposal; reason step-by-step.
""" + AREAS + """End with MODE: recommend or MODE: gap_analysis.

Proposal:
{text}
"""},

    "V4_role_json": {"label": "V4 Role + JSON", "format": "json",
        "system": ROLE_AND_JSON,
        "user": """Task: Assess the methodology of a research proposal.

Proposal:
{text}
"""},

    "V5_hybrid": {"label": "V5 Hybrid", "format": "json",
        "system": ROLE_AND_JSON,
        "user": """Task: Assess the methodology of a research proposal; reason step-by-step.

Proposal:
{text}
"""},
}

def render(template, text):
    return template.replace("{text}", text.strip())

for k, p in PROMPTS.items():
    assert "{text}" in p["user"], f"{k}: user prompt needs a {{text}} placeholder"
    print(f"{p['label']:<22} format={p['format']:<4} chars={len(p['system']) + len(p['user'])}")

V1 Zero-shot           format=text chars=142
V2 Few-shot            format=text chars=2872
V3 Chain-of-thought    format=text chars=666
V4 Role + JSON         format=json chars=2019
V5 Hybrid              format=json chars=2040


In [16]:
# Cell 3 — Test suite (5 real inputs) + run all variations + count tokens (tiktoken)

TEST_SUITE = [
    {"id": "T1", "expected_mode": "gap_analysis", "text":
     """I will fine-tune a pre-trained open-source vision model on a small set of hand-collected
frames of unmodified military vehicles from open-source battlefield drone footage
available via various video-sharing platforms, specifically covering three vehicle classes (i.e., tank, APC, SAM).
Second, I will measure whether its accuracy drops on held-out cage-equipped or camoflaged
vehicles taken from different videos. Third, I will test whether any measured gap narrows
when Stable Diffusion generates realistic variations of 5, 10, and 20 caged or camouflaged
reference images to augment the training set."""},

    # COM SCI-X 455 cap-stone project pitch

    {"id": "T2", "expected_mode": "gap_analysis", "text":
     """The study developed a virtual testbed to evaluate monocular visual odometry (VO) algorithms for
     GPS-denied navigation of small fixed-wing unmanned aircraft systems (SUAS), combining simulated
     flight experiments with real-world flight testing. The virtual environment was developed using the
     AftrBurner visualization engine, incorporating real-world terrain generated from USGS elevation
     data and georeferenced satellite imagery. Simulated aircraft trajectories and realistic flight
     dynamics were generated using the ArduPilot Software-in-the-Loop (SITL) autopilot, while virtual
     camera imagery was produced from a downward-facing monocular camera attached to the aircraft.
     The virtual camera operated at 1280 × 960 resolution, with an 84.872° field of view and a frame
     rate of 31 Hz. Camera calibration was performed using checkerboard images to estimate intrinsic
     parameters and lens distortion under a pinhole camera model, with calibration applicable to both
     simulated and real camera imagery. Three open-source VO algorithms were evaluated: Direct Sparse
     Odometry (DSO), Semi-Direct Visual Odometry (SVO), and ORB-SLAM2, with loop closure disabled in
     ORB-SLAM2 to enable comparison as a visual odometry approach. All algorithms were initially evaluated
     using their default configurations, followed by controlled parameter experiments adjusting image-gradient
     and feature-detection thresholds to improve tracking in low-texture environments and modifying SVO's
     keyframe selection distance. Simulated experiments used a 3.3 km flight trajectory, varying airspeed
     (10 and 25 m/s), maximum aircraft roll rate (25°, 45°, and 65°/s), and altitude (50–400 m in 50 m intervals),
     with environmental conditions and flight trajectories controlled across experiments. Real-world validation
     involved a 1/3-scale Carbon Cub fixed-wing aircraft equipped with a downward-facing Prosilica GT1290
     grayscale camera operating at 1280 × 960 resolution and 33 FPS. Flight data were collected over a 9.6 km
     trajectory at an altitude of 250 m, with airspeeds varying between 20 and 30 m/s. Ground-truth positioning
     and orientation were obtained using a Piksi navigation board providing GPS measurements at 10 Hz and IMU
     measurements at 100 Hz, with sensor data synchronized using a Precision Time Protocol (PTP) server.
     The real-world flight trajectory was also replayed in the virtual environment to compare algorithm
     performance using simulated and actual imagery under corresponding flight conditions. Since monocular
     VO algorithms estimate motion at an unknown relative scale, all experiments incorporated an initial
     straight-and-level flight segment, using the known aircraft altitude at initialization to establish
     the absolute trajectory scale. Algorithm performance was evaluated by comparing estimated
     six-degree-of-freedom trajectories against ground truth, using three-dimensional position Root Mean
     Square Error (RMSE), expressed in metres, and rotation RMSE, expressed in degrees, across complete
     trajectories. The evaluation also recorded initialization failures, tracking losses, and the ability
     to maintain real-time processing at the camera frame rate. The experimental framework enabled systematic
     assessment of robustness under different flight speeds, altitudes, roll rates, image-texture conditions,
     and camera frame rates, while comparison between simulated and real-world imagery examined transferability
     across environments and the effects of image saturation, lighting conditions, sensor noise, and aircraft
     vibrations.
"""},

        # Abstract from Kim, Leishman & Nykl, arXiv 2007.00737

    {"id": "T3", "expected_mode": "gap_analysis", "text":
     """The study introduces WorldFly, a world-model-based Vision-Language-Action (VLA) framework for
     autonomous UAV navigation in complex urban environments, combining future video prediction and
     navigation action generation through a dual-branch coupled architecture. The authors constructed
     a simulation-based benchmark, named Urban Canyon Traversal, using the OpenFly simulation toolchain
     and AirSim urban maps. Skeleton graphs were used to identify street intersections, from which 63
     intersection waypoints were extracted. Expert navigation trajectories were generated using the A\*
     path-planning algorithm, with target waypoints randomly sampled near street ends to encourage longer routes,
     followed by filtering of short-range and invalid trajectories. More than 4,000 navigation trajectories
     were collected, with 100 selected for the TEST-EASY evaluation set and the remaining trajectories used
     for training. An additional 100 trajectories covering 14 newly introduced intersections, strictly excluded
     from training, formed the TEST-HARD set for evaluating generalisation to unseen spatial environments.
     Natural-language navigation instructions were generated automatically using Qwen3-VL, converting expert
     routes and landmark information into textual descriptions. The task was formulated as sequential
     decision-making, with the model receiving three historical first-person-view (FPV) RGB frames and a
     natural-language instruction to predict four future frames and a corresponding sequence of navigation actions.
     Images were processed at 224 × 224 resolution, while instructions were encoded using a T5 text encoder
     with a maximum sequence length of 250 tokens. Future video frames were encoded into latent representations
     using the LTX-Video variational autoencoder (VAE). The proposed architecture consists of two parallel
     branches: a World Model Branch for future video prediction and an Action Expert Branch for navigation action
     generation, connected through bidirectional cross-attention coupling blocks inserted every five transformer
     layers. Both branches were jointly trained using flow matching, with a shared noise timestep to maintain
     temporal alignment between future visual predictions and action sequences. The combined training objective
     weighted the two branch losses equally. Navigation actions were represented as eight-dimensional continuous
     vectors and mapped to ten predefined discrete OpenFly motion primitives, including forward movement, turning,
     vertical movement, lateral movement, and stopping. The experiments compared WorldFly against OpenFly,
     Pi-0-UAV, and a random-action baseline, with all learned models fine-tuned on the same benchmark using
     three historical frames as input. Training was conducted for 40,000 steps using four NVIDIA A100 GPUs
     and a global batch size of 128, with learning rates of 3 × 10⁻⁵ for WorldFly, 5 × 10⁻⁴ for OpenFly,
     and 5 × 10⁻⁵ for Pi-0-UAV. The evaluation used a unified action space, a maximum of 100 steps per episode,
     and a success threshold of reaching within 12 metres of the target. Controlled ablation experiments compared
     the complete WorldFly architecture against a variant without bidirectional coupling, trained using
     the same dataset, optimization schedule, and hyperparameters. Navigation performance was evaluated
     using Success Rate (SR), Navigation Error (NE), and Success weighted by Path Length (SPL).
     Additional evaluation included stratifying trajectories into short, medium, and long categories,
     each containing 30 tasks, and assessing future video prediction quality using Peak Signal-to-Noise
     Ratio (PSNR), Structural Similarity Index Measure (SSIM), and Learned Perceptual Image Patch Similarity
     (LPIPS). Computational efficiency was evaluated through inference latency and peak GPU memory consumption,
     measured on a single NVIDIA A100 GPU using a 50-step flow-matching denoising procedure. Generalisation
     was assessed by comparing performance between TEST-EASY, containing previously encountered intersection
     layouts, and TEST-HARD, containing entirely unseen intersections and spatial geometries. The evaluation
     therefore examined transfer to unfamiliar urban environments, changes in viewpoint, visual occlusions,
     and increasingly long navigation trajectories.
"""},

    # LLM summarized methodology from Zheng et al., WorldFly, arXiv 2606.06147

    {"id": "T4", "expected_mode": "gap_analysis", "text":
     """The study proposes YOLOatr, a modified YOLOv5s architecture for automatic target detection,
     classification, and localization in mid-wave thermal infrared (MWIR) imagery. The authors used
     the publicly available Defense Systems Information Analysis Center (DSIAC) MWIR ATR dataset,
     which contains thermal infrared imagery of tactical and civilian vehicles recorded in a desert environment.
     The original dataset comprises approximately 367,200 frames extracted from one-minute videos recorded at
     30 FPS, covering 13 target types, distances ranging from 1,000 to 5,000 metres at 500-metre intervals,
     72 aspect angles, and daytime and nighttime conditions. Four vehicle categories were selected for the
     experiments: T-72 tank, BRDM2, pickup truck, and SUV, with 3,600 images per category, resulting in a dataset
     of 14,400 images. The images were annotated with ground-truth bounding boxes. The dataset was divided into
     training (70%), validation (20%), and testing (10%) subsets. Two experimental testing protocols were
     established to evaluate generalisation across target distances. In the correlated testing protocol (T1),
     training and testing images covered the same distance ranges, whereas in the decorrelated testing protocol
     (T2), the model was evaluated at distances not represented during training. Two dataset configurations were
     considered: DS1, trained using images from 1.0–2.5 km and evaluated at the same distances under T1 and at
     3.0 km under T2; and DS2, trained using images from 3.0–4.5 km and evaluated at the same distances under T1
     and at 5.0 km under T2. The proposed YOLOatr architecture was developed by modifying the YOLOv5s single-stage,
     anchor-based object detector. The modifications included adding a P2 detection head to improve the detection
     of small objects and replacing the original PANet neck with a BiFPN (Weighted Bidirectional Feature Pyramid
     Network) to enhance multi-scale feature fusion. Controlled experiments compared training from scratch with
     transfer learning using ImageNet-pretrained weights. Training from scratch involved 100 epochs with randomly
     initialized weights, while transfer learning involved freezing the backbone for 30 epochs followed by 10 epochs
     of fine-tuning with all layers unfrozen. The authors also investigated different data augmentation configurations,
     comparing their effects on training and validation performance and developing a custom augmentation profile
     (CAP). The final profile incorporated brightness and contrast variations, rotation, translation, scaling,
     perspective transformation, horizontal and vertical flipping, mosaic, mix-up, and copy-paste augmentation,
     alongside adjustments to hue, saturation, value, and focal loss. All experiments used an input resolution of
     640 × 640 pixels, a batch size of 32, stochastic gradient descent (SGD), and a final training budget of 100
     epochs. Experiments were conducted using Google Colab Pro with an NVIDIA Tesla P100 GPU (16 GB) and 32 GB RAM.
     The original YOLOv5s architecture served as the primary baseline, with YOLOatr evaluated under comparable
     experimental conditions. The authors also contextualised their results against previously published detection
     and recognition approaches evaluated on the DSIAC dataset, including YOLOv3, Faster R-CNN, and Mask R-CNN.
     Evaluation metrics included Precision, Recall, and mean Average Precision at an IoU threshold of 0.5 (mAP@0.5).
     Additional evaluation involved Precision–Recall curves, confusion matrices, inference time, frames per second
     (FPS), model parameter count, and computational complexity measured in GFLOPs. Performance was assessed both
     overall and separately for the four vehicle categories under the correlated and decorrelated testing protocols.
     Qualitative examination of detection outputs was also conducted to assess predictions in cluttered backgrounds
     and at long target distances, including 5,000 metres. The generalisation experiments investigated the effects
     of changes in target distance and scale between training and testing data, with evaluation covering different
     aspect angles, background conditions, and daytime and nighttime imagery.
"""},
        # LLM summarized methodology from Safdar et al., YOLOatr, arXiv 2507.11267

    {"id": "T5", "expected_mode": "gap_analysis", "text":
     """The study adopted a comparative experimental approach to investigate the performance of four
     YOLO-based object detection architectures (YOLOv5, YOLOv8, YOLOv9, and YOLOv11) in recognizing
     military vehicles in aerospace imagery. The authors developed a custom dataset of 2,274 images
     collected from publicly accessible web resources, open-access platforms, satellite imagery,
     and UAV-style aerial photographs. Object annotation was performed using the Computer Vision Annotation Tool
     (CVAT), with each target assigned a class label and bounding box. The original annotation scheme contained
     eight classes, which were subsequently harmonized by removing three irrelevant categories and merging
     duplicate labels, resulting in four target classes: military-vehicle, tank, truck, and jeep, with a
     total of 11,454 annotated object instances. The images were resized to a common resolution of
     640 × 640 pixels, and the dataset was divided into training (1,592 images), validation (341 images), and
     testing (341 images) subsets, corresponding to an approximate 70/15/15 split. The same dataset partition
     was maintained across all four architectures to ensure consistent experimental comparisons.
     The models were trained independently under a common experimental protocol using 100 epochs and the same
     input resolution. The comparison focused on differences between the architectures, which share a
     single-stage object detection framework but employ different network components: CSP-based feature extraction
     in YOLOv5, C2f modules and an anchor-free detection head in YOLOv8, PGI and GELAN components in YOLOv9,
     and C3k2 and C2PSA modules in YOLOv11. Model performance was assessed quantitatively using Precision,
     Recall, F1-score, Average Precision (AP), mAP@50 (at an Intersection over Union threshold of 0.50),
     and mAP@50–95 (across IoU thresholds from 0.50 to 0.95), alongside inference speed measured in frames
     per second (FPS). Normalized confusion matrices were also examined to derive class-wise Recall from their
     diagonal values, while Precision–Confidence curves were generated by varying detection-confidence
     thresholds and recalculating precision. In addition to quantitative benchmarking, qualitative evaluation was
     conducted on previously unseen aerospace images representing different environmental conditions, including
     forested areas with partially occluded vehicles, open grassland, unpaved roads, and mixed settlement and
     floodplain environments. These representative scenes were used to compare detection outputs, localization,
     classification, and false-positive occurrences across the four architectures. The evaluation also considered
     the distribution of annotated instances across the dataset classes, with the generic military-vehicle category
     accounting for 11,312 instances (98.76%), compared with 76 tanks, 42 trucks, and 24 jeeps. The experimental
     design therefore provided a controlled comparison of four YOLO architectures using identical data partitions,
     image resolution, and training duration, combining aggregate detection metrics, class-specific recognition
     analysis, and qualitative examination of model predictions.

"""},
]
    # LLM summarized methodology from Kashtan & Hnatushenko, CEUR Workshop Proceedings, Vol-4300 (2026)

assert len(TEST_SUITE) == 5, "Need exactly 5 test inputs"

def call_model(system, user):
    msgs = ([{"role": "system", "content": system}] if system.strip() else []) + [{"role": "user", "content": user}]
    r = client.chat.completions.create(model=MODEL, messages=msgs)
    return r.choices[0].message.content or ""

def strip_fences(s):
    return re.sub(r"^```(?:json)?\s*|\s*```$", "", s.strip())

def check(out, fmt):
    """Returns (format_pass, detected_mode). Format checked for JSON only; text -> NaN."""
    if fmt == "json":
        try:
            return 1.0, Assessment.model_validate_json(strip_fences(out)).mode
        except (ValidationError, ValueError):
            try:
                return 0.0, json.loads(strip_fences(out)).get("mode")
            except Exception:
                return 0.0, None
    m = re.search(r"MODE:\s*\**\s*(recommend|gap[_ ]analysis)", out, re.I)
    return float("nan"), (m.group(1).lower().replace(" ", "_") if m else None)

records = []
for vkey, p in PROMPTS.items():
    for case in TEST_SUITE:
        if case["text"].startswith("PASTE"):
            continue  # skip placeholders
        user = render(p["user"], case["text"])
        try:
            out = call_model(p["system"], user)
        except Exception as e:
            print(f"ERROR {vkey}/{case['id']}: {e}"); continue
        ok, mode = check(out, p["format"])
        in_tok, out_tok = count_tokens(p["system"]) + count_tokens(user), count_tokens(out)
        records.append({
            "variation": vkey, "label": p["label"], "test_id": case["id"],
            "expected_mode": case["expected_mode"], "detected_mode": mode,
            "format_pass": ok,
            "input_tokens": in_tok, "output_tokens": out_tok, "total_tokens": in_tok + out_tok,
            "output": out,
        })
        fmt_str = "N/A" if p["format"] == "text" else ("PASS" if ok else "FAIL")
        print(f"{p['label']:<22} {case['id']}  fmt={fmt_str}  mode={mode}  tokens={in_tok + out_tok}")

raw = pd.DataFrame(records)
raw.to_csv("results_raw.csv", index=False)

def show(vkey, test_id):
    """Read one output for grading, e.g. show('V3_cot', 'T1')"""
    r = raw[(raw.variation == vkey) & (raw.test_id == test_id)].iloc[0]
    print(f"=== {r.label} | {test_id} | mode={r.detected_mode} ({r.expected_mode}) | {r.total_tokens} tok ===\n")
    print(r.output)

<>:62: SyntaxWarning: invalid escape sequence '\*'
<>:62: SyntaxWarning: invalid escape sequence '\*'
/tmp/ipykernel_30278/2065729956.py:62: SyntaxWarning: invalid escape sequence '\*'
  intersection waypoints were extracted. Expert navigation trajectories were generated using the A\*


V1 Zero-shot           T1  fmt=N/A  mode=gap_analysis  tokens=688
V1 Zero-shot           T2  fmt=N/A  mode=gap_analysis  tokens=1301
V1 Zero-shot           T3  fmt=N/A  mode=gap_analysis  tokens=1527
V1 Zero-shot           T4  fmt=N/A  mode=recommend  tokens=1432
V1 Zero-shot           T5  fmt=N/A  mode=recommend  tokens=1220
V2 Few-shot            T1  fmt=N/A  mode=gap_analysis  tokens=1348
V2 Few-shot            T2  fmt=N/A  mode=gap_analysis  tokens=1961
V2 Few-shot            T3  fmt=N/A  mode=gap_analysis  tokens=2090
V2 Few-shot            T4  fmt=N/A  mode=gap_analysis  tokens=2128
V2 Few-shot            T5  fmt=N/A  mode=gap_analysis  tokens=1865
V3 Chain-of-thought    T1  fmt=N/A  mode=gap_analysis  tokens=1110
V3 Chain-of-thought    T2  fmt=N/A  mode=gap_analysis  tokens=1738
V3 Chain-of-thought    T3  fmt=N/A  mode=recommend  tokens=1890
V3 Chain-of-thought    T4  fmt=N/A  mode=gap_analysis  tokens=1893
V3 Chain-of-thought    T5  fmt=N/A  mode=gap_analysis  tokens=1682
V4 Ro

In [17]:
# Print prompt outputs per model version:
for _, r in raw.iterrows():
    print(f"\n{'='*80}\n{r.label} | {r.test_id} | mode={r.detected_mode} (expected {r.expected_mode}) | {r.total_tokens} tok\n{'='*80}")
    print(r.output)


V1 Zero-shot | T1 | mode=gap_analysis (expected gap_analysis) | 688 tok
MODE: gap_analysis

The proposed research methodology presents an interesting approach to fine-tuning a vision model for military vehicle classification but contains several gaps that could affect the validity and applicability of the results. Here are some critical points for consideration:

1. **Dataset Composition and Size**: The methodology mentions using a "small set of hand-collected frames." While hand-collecting data may ensure specificity and quality, the limited size may introduce bias and limit the model's generalization capabilities. Discussing the exact number of images and how they will be selected could help in assessing the representativeness of the dataset.

2. **Diversity of Sources**: The proposal states that the training data is sourced from "various video-sharing platforms," but it does not clarify the criteria for selecting these videos or the demographic diversity regarding the vehicle's con

In [22]:
# Cell 4 — LLM-judged Correctness

import pandas as pd
pd.set_option("display.max_colwidth", None)

PRICE_IN_PER_1M  = 0.15      # USD per 1M input tokens (model under test)
PRICE_OUT_PER_1M = 0.60      # USD per 1M output tokens
JUDGE_MODEL      = "gpt-4o"  # stronger than the model under test, to avoid self-grading
REJUDGE          = True      # True = re-score all outputs; set back to False after one run

TEST_IDS = [c["id"] for c in TEST_SUITE]

# 1–5 Usefulness scores per input, order T1..T5. None = not yet graded.
USEFULNESS = {
    "V1_zero_shot": [3.3, 3.3, 3.3, 3.3, 3.3],
    "V2_few_shot":  [3.5, 3.5, 3.5, 3.5, 3.5],
    "V3_cot":       [4, 4, 4, 4, 4],
    "V4_role_json": [3.5, 3.5, 3.5, 3.5, 3.5],
    "V5_hybrid":    [3.5, 3.5, 3.5, 3.5, 3.5],
}

# Optional: reference gaps per input to anchor the judge ("" = judge unaided)
ANSWER_KEY = {
    "T1": "Train/test leakage if frames from the same video or vehicle end up in both sets (split by video, deduplicate reposted clips); "
          "small hand-collected dataset with no stated size, class balance or labelling check; no baseline comparison; "
          "Stable Diffusion images may not resemble real cages or camouflage (realism not validated, and generating from few reference images risks near-copies); "
          "no metric beyond accuracy and no repeated runs or confidence intervals.",
    "T2": "",
    "T3": "",
    "T4": "",
    "T5": "",
}

# Optional: override a judge score after reading it, e.g. {("V2_few_shot", "T3"): 2}
CORRECTNESS_OVERRIDES = {}

# Format pass for text variations: per-input hand checks [1,0,...] or None -> N/A
FORMAT_PASS = {"V1_zero_shot": None, "V2_few_shot": None, "V3_cot": None,
               "V4_role_json": None, "V5_hybrid": None}

# LLM judge: Correctness
CORRECTNESS_RUBRIC = """Score the CORRECTNESS of a methodology assessment from 1 to 5.
Judge only whether it is RIGHT, not how well written or actionable it is.

Check:
- Gaps listed are real weaknesses of the stated method (not invented, not already addressed in the input).
- Important gaps are not missed.
- Claims about the input and about methods are factually accurate.
- Things merely not stated in a short text are framed as "unspecified/verify", not as confirmed flaws.

5 = all major gaps found, nothing false
4 = misses one minor gap or has one small inaccuracy
3 = misses a major gap or contains a clear error
2 = several misses or a significant error, or the recommended method is unsound
1 = mostly wrong

{key_block}INPUT GIVEN TO THE AI:
{input_text}

AI OUTPUT:
{output}

Return ONLY JSON: {{"score": 1-5, "reason": "one sentence"}}"""

texts = {c["id"]: c["text"] for c in TEST_SUITE}

def judge_correctness(test_id, output):
    key = ANSWER_KEY.get(test_id, "").strip()
    key_block = f"REFERENCE GAPS (guide, not exhaustive):\n{key}\n\n" if key else ""
    prompt = CORRECTNESS_RUBRIC.format(key_block=key_block, input_text=texts[test_id], output=output)
    r = client.chat.completions.create(model=JUDGE_MODEL, temperature=0,
                                       response_format={"type": "json_object"},
                                       messages=[{"role": "user", "content": prompt}])
    try:
        j = json.loads(r.choices[0].message.content)
        return float(j["score"]), j.get("reason", "")
    except Exception:
        return float("nan"), "judge output unparseable"

if REJUDGE or "correctness" not in raw.columns:
    raw["correctness"], raw["correctness_reason"] = float("nan"), ""
for i in raw.index[raw["correctness"].isna()]:
    row = raw.loc[i]
    s, why = judge_correctness(row.test_id, row.output)
    raw.loc[i, "correctness"], raw.loc[i, "correctness_reason"] = s, why
    print(f"judged {row.label:<22} {row.test_id}  {s}")

for (v, t), s in CORRECTNESS_OVERRIDES.items():
    m = (raw.variation == v) & (raw.test_id == t)
    raw.loc[m, "correctness"], raw.loc[m, "correctness_reason"] = s, "[manual override]"
raw.to_csv("results_raw.csv", index=False)

# Results table
def avg(xs):
    xs = [x for x in (xs or []) if x is not None]
    return sum(xs) / len(xs) if xs else float("nan")

raw["cost_usd"] = (raw.input_tokens * PRICE_IN_PER_1M + raw.output_tokens * PRICE_OUT_PER_1M) / 1e6
auto = raw.groupby("variation").agg(fmt=("format_pass", "mean"),
                                    corr=("correctness", "mean"),
                                    tok=("total_tokens", "mean"),
                                    cost=("cost_usd", "mean"),
                                    n=("test_id", "nunique"))

def fmt_rate(v):
    manual = FORMAT_PASS.get(v)
    if manual is not None:
        return avg(manual) * 100
    if PROMPTS[v]["format"] == "json":
        return auto.loc[v, "fmt"] * 100
    return float("nan")

def reasons(v):
    r = raw[raw.variation == v].sort_values("test_id")
    return "\n".join(f"{t} ({s:.0f}): {why}" for t, s, why in
                     zip(r.test_id, r.correctness, r.correctness_reason))

variations = [v for v in PROMPTS if v in auto.index]
table = pd.DataFrame([{
    "Variation":          PROMPTS[v]["label"],
    "Avg Correctness":    auto.loc[v, "corr"],
    "Format Pass Rate":   fmt_rate(v),
    "Avg Usefulness":     avg(USEFULNESS.get(v)),
    "Avg Tokens":         auto.loc[v, "tok"],
    "Cost/1K calls":      auto.loc[v, "cost"] * 1000,
    "Correctness Reason": reasons(v),
} for v in variations])

display(table.style
        .format({"Avg Correctness": "{:.2f}", "Format Pass Rate": "{:.0f}%", "Avg Usefulness": "{:.2f}",
                 "Avg Tokens": "{:.0f}", "Cost/1K calls": "${:.3f}"}, na_rep="N/A")
        .set_properties(subset=["Correctness Reason"],
                        **{"white-space": "pre-wrap", "text-align": "left", "min-width": "500px"})
        .hide(axis="index"))

judged V1 Zero-shot           T1  3.0
judged V1 Zero-shot           T2  4.0
judged V1 Zero-shot           T3  5.0
judged V1 Zero-shot           T4  4.0
judged V1 Zero-shot           T5  4.0
judged V2 Few-shot            T1  5.0
judged V2 Few-shot            T2  4.0
judged V2 Few-shot            T3  4.0
judged V2 Few-shot            T4  4.0
judged V2 Few-shot            T5  4.0
judged V3 Chain-of-thought    T1  4.0
judged V3 Chain-of-thought    T2  4.0
judged V3 Chain-of-thought    T3  4.0
judged V3 Chain-of-thought    T4  4.0
judged V3 Chain-of-thought    T5  4.0
judged V4 Role + JSON         T1  4.0
judged V4 Role + JSON         T2  3.0
judged V4 Role + JSON         T3  5.0
judged V4 Role + JSON         T4  3.0
judged V4 Role + JSON         T5  5.0
judged V5 Hybrid              T1  4.0
judged V5 Hybrid              T2  3.0
judged V5 Hybrid              T3  5.0
judged V5 Hybrid              T4  3.0
judged V5 Hybrid              T5  5.0


Variation,Avg Correctness,Format Pass Rate,Avg Usefulness,Avg Tokens,Cost/1K calls,Correctness Reason
V1 Zero-shot,4.00,N/A,3.30,1234,$0.436,"T1 (3): Misses major gaps like train/test leakage and lacks baseline comparison. T2 (4): Misses one minor gap regarding the potential impact of varying weather conditions on VO performance. T3 (5): All major gaps were identified, and no false claims were made. T4 (4): The assessment is mostly correct but misses a minor gap in not specifying how augmentation effectiveness is quantitatively assessed. T5 (4): The assessment is mostly correct but misses a minor gap regarding the lack of details on data augmentation techniques."
V2 Few-shot,4.20,N/A,3.50,1878,$0.504,"T1 (5): All major gaps are identified accurately, and there are no false claims or significant omissions. T2 (4): Misses one minor gap regarding the lack of exploration of varying environmental conditions in real-world tests. T3 (4): The assessment is mostly correct but misses a minor gap regarding the potential impact of using a simulation-based benchmark on real-world applicability. T4 (4): The assessment is mostly correct but misses a minor gap regarding the potential impact of the specific augmentation techniques used on the model's performance. T5 (4): Misses one minor gap regarding the potential impact of image resizing on model performance."
V3 Chain-of-thought,4.00,N/A,4.00,1663,$0.642,"T1 (4): The assessment is mostly correct but misses the specific issue of train/test leakage from frames of the same video or vehicle. T2 (4): The assessment is mostly correct but misses minor details such as the explicit mention of statistical significance and the number of experimental repetitions. T3 (4): The assessment is mostly correct but misses minor details such as the lack of confidence intervals and repeated runs, which are important for statistical significance. T4 (4): The assessment is mostly correct but misses minor details on data leakage prevention and confounding factors. T5 (4): The assessment is mostly correct but misses a minor gap regarding the lack of mention of potential overfitting due to the small dataset size."
V4 Role + JSON,4.00,60%,3.50,1452,$0.363,"T1 (4): Misses the potential issue of train/test leakage but otherwise accurately identifies gaps and unspecified areas. T2 (3): The analysis incorrectly assumes data splitting is relevant, missing the context of VO evaluation, and generalization concerns are partially addressed. T3 (5): All major gaps were identified correctly, and no false claims were made. T4 (3): The assessment misses a major gap by not addressing potential overfitting due to the limited number of vehicle categories and lacks mention of the need to verify the impact of the custom augmentation profile. T5 (5): All major gaps found, nothing false."
V5 Hybrid,4.00,80%,3.50,1481,$0.378,"T1 (4): The assessment is mostly correct but misses the specific issue of Stable Diffusion potentially generating near-copies of reference images. T2 (3): The assessment contains a clear error regarding data leakage, as the input does not mention training the algorithms, only evaluation. T3 (5): All major gaps were identified correctly, and no false claims were made. T4 (3): The assessment misses a major gap by not addressing potential biases in the dataset selection and does not mention the need to verify the robustness of the model across different environmental conditions. T5 (5): All major gaps were identified correctly, and no false claims were made."


Winning variation: I found the V5 hybrid to be the best bang for the buck. While I preferred CoT, being a novice and therefore benefiting from step-by-step explanation, it was almost twice the cost.

---



In [25]:
# Cell 5 — Part 2: Adversarial test of the worst-performing variation
WORST = 2    # From previous cell (combo of lower correctness/usefulness and higher cost)
JUDGE_MODEL = "gpt-4o"

vkey = list(PROMPTS)[WORST - 1]
p = PROMPTS[vkey]
print(f"Testing: {p['label']}\n")

ADV_SUITE = [
    # Ambiguous instruction: "the approach" is unclear — two approaches are given
    {"id": "A1",
     "text": """I want to find out whether cage armor and camouflage reduce the performance of open-source
vehicle detectors on drone footage. One option is to fine-tune YOLOv8 on about 300 labelled frames of
unmodified tanks and test it on caged ones. The other is to run CLIP zero-shot with text prompts such as
"a tank with cage armor". Evaluate the approach and improve it."""},

    # Missing format: method given only as fragmented notes
    {"id": "A2",
     "text": """cage armor detection
- yolo? or clip zero-shot
- footage from telegram channels, maybe 200 imgs
- tanks/ifv/trucks
- test on caged ones
- synthetic data later??
acc metric"""},

    # Conflicting constraints
    {"id": "A3",
     "text": """I will fine-tune a 70B-parameter open-source vision-language model to classify 10 vehicle
types, including caged and camouflaged variants, from drone footage. I have no labelled data and
will not label any. Training must run on my laptop CPU, be completed within one week, use only free
tools, and achieve at least 99% accuracy on unseen footage, which I will measure on the training images."""},
]

# ---------- Judge rubrics (one per failure mode) ----------
ADV_RUBRICS = {
    "A1": (
        "The input describes TWO alternative approaches (fine-tuned YOLOv8 vs. CLIP zero-shot) and asks to "
        "'evaluate the approach', which is ambiguous.\n"
        "PASS: recognises there are two approaches and assesses both (or explicitly states which one it assumes "
        "and why), identifying real gaps in each, e.g. only 300 frames of one class, no caged examples in training, "
        "no baseline, no split detail, prompt sensitivity of CLIP.\n"
        "PARTIAL: assesses both but superficially, or assesses one and only mentions the other.\n"
        "FAIL: silently assesses only one, or merges them into one incoherent method."
    ),
    "A2": (
        "The input is unstructured notes implying a method (YOLO or CLIP zero-shot, Telegram footage, "
        "~200 images, tanks/IFV/trucks, test on caged vehicles, synthetic data later, accuracy metric).\n"
        "PASS: treats it as an existing method, uses nearly all fragments, and flags the hedged or unspecified "
        "items (undecided model, sample size, metric, split, synthetic data plan) as gaps.\n"
        "PARTIAL: treats it as a method but drops several fragments or flags little of the vagueness.\n"
        "FAIL: treats it as having no methodology, or ignores most fragments."
    ),
    "A3": (
        "The input has mutually incompatible constraints: 70B model on a laptop CPU, supervised fine-tuning with "
        "no labels and a refusal to label, 1 week, 99% accuracy on 10 classes incl. caged/camouflaged, and "
        "evaluation on the training images.\n"
        "PASS: identifies the conflicts (compute, labels, accuracy target) and the train-set evaluation leakage as "
        "serious gaps, AND its recommended methodology is actually feasible under the constraints it did not tell "
        "the user to relax (e.g. zero-shot classification, or explicitly requires some labelling).\n"
        "PARTIAL: identifies most conflicts/leakage but the recommendation still violates a constraint, or the "
        "unrealistic accuracy target goes unchallenged.\n"
        "FAIL: accepts the constraints and proposes a plan that cannot work, or misses the leakage."
    ),
}

def adv_judge(case_id, input_text, output):
    prompt = (f"You are grading whether an AI's methodology assessment resisted an adversarial input.\n\n"
              f"RUBRIC:\n{ADV_RUBRICS[case_id]}\n\nINPUT GIVEN TO THE AI:\n{input_text}\n\n"
              f"AI OUTPUT:\n{output}\n\n"
              'Return ONLY JSON: {"verdict": "PASS" | "PARTIAL" | "FAIL", "reason": "one sentence"}')
    r = client.chat.completions.create(model=JUDGE_MODEL, temperature=0,
                                       response_format={"type": "json_object"},
                                       messages=[{"role": "user", "content": prompt}])
    try:
        j = json.loads(r.choices[0].message.content)
        return j.get("verdict"), j.get("reason", "")
    except Exception:
        return None, "judge output unparseable"

# ---------- Run ----------
adv_records = []
for case in ADV_SUITE:
    user = render(p["user"], case["text"])
    try:
        out = call_model(p["system"], user)
    except Exception as e:
        print(f"ERROR {case['id']}: {e}"); continue
    ok, _ = check(out, p["format"])
    verdict, reason = adv_judge(case["id"], case["text"], out)
    in_tok, out_tok = count_tokens(p["system"]) + count_tokens(user), count_tokens(out)
    adv_records.append({
        "test_id": case["id"], "format_pass": ok, "total_tokens": in_tok + out_tok,
        "verdict": verdict, "reason": reason, "output": out,
    })
    print("=" * 90)
    print(f"{case['id']} | judge={verdict} | tokens={in_tok + out_tok}")
    print("-" * 90)
    print(out, "\n")

adv = pd.DataFrame(adv_records)

# ---------- Manual override after reading outputs, e.g. {"A3": ("PARTIAL", "recommendation still needs labels")} ----------
ADV_OVERRIDES = {}
for t, (v, note) in ADV_OVERRIDES.items():
    adv.loc[adv.test_id == t, ["verdict", "reason"]] = [v, f"[manual] {note}"]

adv.to_csv("results_adversarial.csv", index=False)

pd.set_option("display.max_colwidth", None)
display(adv[["test_id", "format_pass", "total_tokens", "verdict", "reason"]]
        .style.format({"format_pass": "{:.0%}"}, na_rep="N/A")
        .set_properties(**{"text-align": "left", "vertical-align": "top"})
        .set_properties(subset=["reason"], **{"white-space": "pre-wrap", "min-width": "500px"})
        .hide(axis="index"))

Testing: V2 Few-shot

A1 | judge=FAIL | tokens=1320
------------------------------------------------------------------------------------------
MODE: gap_analysis  
RESEARCH QUESTION: Do cage armor and camouflage reduce the performance of open-source vehicle detectors on drone footage?  

1. DATA AND ANNOTATION: [GAP | HIGH] Using only 300 labelled frames may not be sufficient to train or fine-tune a model effectively. Additionally, labeling consistency should be ensured. Fix: Collect a larger, balanced dataset of various tank types with and without armor and camouflage features across multiple scenarios and environmental conditions, involving multiple annotators to ensure labeling reliability.

2. TRAIN/VAL/TEST SPLIT: [GAP | HIGH] If the split does not account for diversity in the dataset, it can limit generalizability. Fix: Implement a stratified split based on different visual conditions (e.g., lighting, angles, and backgrounds) to ensure a representative distribution in all sets.



test_id,format_pass,total_tokens,verdict,reason
A1,N/A,1320,FAIL,The AI output only assesses the YOLOv8 approach and does not mention or evaluate the CLIP zero-shot approach.
A2,N/A,1251,PASS,"The AI output treats the input as an existing method, uses nearly all fragments, and flags the hedged or unspecified items as gaps."
A3,N/A,1250,FAIL,"The AI output accepts the constraints and proposes a plan that cannot work, as it suggests collecting and labeling data, which violates the user's refusal to label any data."


In [26]:
# Cell 6 — Part 3: Repaired prompt vs original, graded by LLM judge

# Repair: one check per failure mode
REPAIR_PROCEDURE = """Before answering, work through these checks in order.

CHECK 1 — Is the instruction ambiguous?
- If the text describes more than one approach, or the request does not say which part to assess,
  do NOT silently pick one. Assess each approach separately across the six areas, or state explicitly
  which one you assume and why. Then say which approach is stronger and why.

CHECK 2 — Is the input fragmented or unstructured (notes, bullets, half-sentences)?
- If yes, first rewrite it as one coherent method statement using EVERY fragment. Do not drop any.
- Every hedge ("?", "maybe", "or", "later") and every unspecified item (sample size, metric,
  train/test split, labelling) is a gap and must be listed.

CHECK 3 — Do the stated constraints conflict?
- Test every pair of constraints (data, labels, compute, model size, time, cost, accuracy target,
  evaluation set). List each pair that cannot hold at the same time.
- Each conflict is a HIGH-severity gap. For each, state which constraint must be relaxed.
- Evaluating on training data is always a HIGH-severity gap (data leakage).
- Verify your recommended methodology satisfies every constraint you did NOT tell the user to relax.
  If no feasible plan exists, say so.

Before the six areas, add:
ASSUMPTIONS / RECONSTRUCTION: (from Checks 1–2; "none" if not needed)
CONSTRAINT CONFLICTS: (from Check 3; "none" if none)
"""

REPAIR_JSON_NOTE = """
In JSON, put assumptions, the reconstructed method and constraint conflicts at the start of
"recommended_methodology", and list each conflict as a "gap" in the relevant area.
"""

repaired = dict(p)
repaired["label"] = p["label"] + " (repaired)"
if p["format"] == "json":
    repaired["system"] = (p["system"] or "") + "\n\n" + REPAIR_PROCEDURE + REPAIR_JSON_NOTE
else:
    u = p["user"]
    i = u.rfind("Input: {text}")
    if i == -1:
        i = u.rfind("{text}")
    repaired["user"] = u[:i] + REPAIR_PROCEDURE + "\n" + u[i:]

# ---------- Run ----------
def run(prompt, cases, tag):
    rows = []
    for case in cases:
        if case["text"].startswith("PASTE"):
            continue
        user = render(prompt["user"], case["text"])
        try:
            out = call_model(prompt["system"], user)
        except Exception as e:
            print(f"ERROR {tag}/{case['id']}: {e}"); continue
        ok, _ = check(out, prompt["format"])
        if case["id"] in ADV_RUBRICS:
            verdict, reason = adv_judge(case["id"], case["text"], out)
            score = None
        else:
            score, reason = judge_correctness(case["id"], out)
            verdict = None
        in_tok, out_tok = count_tokens(prompt["system"]) + count_tokens(user), count_tokens(out)
        rows.append({"version": tag, "test_id": case["id"], "verdict": verdict, "correctness": score,
                     "reason": reason, "format_pass": ok, "total_tokens": in_tok + out_tok, "output": out})
    return rows

RUN_REGRESSION = True   # also re-run Part 1 inputs to confirm the repair doesn't hurt them
cases = ADV_SUITE + (TEST_SUITE if RUN_REGRESSION else [])
comp = pd.DataFrame(run(p, cases, "original") + run(repaired, cases, "repaired"))

# Manual override after reading outputs, e.g. {("A3", "repaired"): ("PARTIAL", "still needs labels")}
OVERRIDES = {}
for (tid, tag), (v, note) in OVERRIDES.items():
    m = (comp.test_id == tid) & (comp.version == tag)
    comp.loc[m, "verdict"], comp.loc[m, "reason"] = v, f"[manual] {note}"

comp.to_csv("results_repaired_vs_original.csv", index=False)

# Side-by-side outputs (adversarial only)
for tid in [c["id"] for c in ADV_SUITE]:
    for tag in ["original", "repaired"]:
        r = comp[(comp.test_id == tid) & (comp.version == tag)]
        if r.empty: continue
        r = r.iloc[0]
        print("=" * 90)
        print(f"{tid} | {tag.upper()} | judge={r.verdict} | tokens={r.total_tokens}")
        print(f"Reason: {r.reason}")
        print("-" * 90)
        print(r.output, "\n")

# Table 1: adversarial, original vs repaired
ADV_TYPES = {"A1": "Ambiguous instruction", "A2": "Missing format", "A3": "Conflicting constraints"}
a = comp[comp.test_id.isin(ADV_RUBRICS)]
o, rp = a[a.version == "original"].set_index("test_id"), a[a.version == "repaired"].set_index("test_id")
adv_cmp = pd.DataFrame({
    "Type":              [ADV_TYPES[t] for t in o.index],
    "Original verdict":  o.verdict,
    "Repaired verdict":  rp.verdict,
    "Original tokens":   o.total_tokens,
    "Repaired tokens":   rp.total_tokens,
    "Repaired reason":   rp.reason,
}).rename_axis("Test").reset_index()

pd.set_option("display.max_colwidth", None)
display(adv_cmp.style
        .set_properties(**{"text-align": "left", "vertical-align": "top"})
        .set_properties(subset=["Repaired reason"], **{"white-space": "pre-wrap", "min-width": "450px"})
        .hide(axis="index"))

# Table 2: regression on Part 1 inputs
if RUN_REGRESSION:
    reg = (comp[~comp.test_id.isin(ADV_RUBRICS)]
           .groupby("version")
           .agg(avg_correctness=("correctness", "mean"), avg_tokens=("total_tokens", "mean"))
           .round(2))
    display(reg)

A1 | ORIGINAL | judge=FAIL | tokens=1303
Reason: The AI output only assesses the YOLOv8 approach and does not mention or evaluate the CLIP zero-shot approach.
------------------------------------------------------------------------------------------
MODE: gap_analysis  
RESEARCH QUESTION: Do cage armor and camouflage reduce the performance of open-source vehicle detectors on drone footage?  
1. DATA AND ANNOTATION: [GAP | HIGH] 300 frames may not provide sufficient variability or representativeness for effective model training. Fix: Collect a larger dataset with diverse examples of both unmodified and modified tanks, ensuring variations in lighting, angles, and distance. Label all frames using multiple annotators to ensure accuracy and inter-annotator agreement.
  
2. TRAIN/VAL/TEST SPLIT: [GAP | HIGH] Random splitting may lead to data leakage, where test frames are too similar or identical to training frames. Fix: Split the dataset based on distinct operational conditions, such as tim

Test,Type,Original verdict,Repaired verdict,Original tokens,Repaired tokens,Repaired reason
A1,Ambiguous instruction,FAIL,PASS,1303,2127,"The AI output recognizes and assesses both approaches separately, identifying specific gaps and suggesting improvements for each."
A2,Missing format,PASS,PASS,1290,1789,"The AI treats the input as an existing method, uses nearly all fragments, and flags the hedged or unspecified items as gaps."
A3,Conflicting constraints,PARTIAL,PARTIAL,1244,1842,The AI identifies most conflicts and leakage but does not explicitly challenge the unrealistic accuracy target.


,avg_correctness,avg_tokens
version,,
original,4.2,1906.8
repaired,3.8,2500.6


In [ ]:
r = comp[(comp.test_id == "A3") & (comp.version == "repaired")].iloc[0]
o = r.output.lower()
print("1. mode ok:      ", r.detected_mode, "->", r.detected_mode == "gap_analysis")
print("2. conflict word:", [w for w in ["conflict","infeasible","not feasible","impossible","cannot","unrealistic"] if w in o])
print("3. leakage word: ", [w for w in ["leak","training images","training data","same data","held-out","held out"] if w in o])
print("\n", r.output)

1. mode ok:       gap_analysis -> True
2. conflict word: []
3. leakage word:  ['training images']

 {
  "mode": "gap_analysis",
  "research_question": "How can we classify 10 vehicle types from drone footage using a vision-language model?",
  "methodology_vague": false,
  "vague_elements": [
    "no labelled data",
    "will not label any",
    "achieve at least 99% accuracy on unseen footage",
    "measure on the training images"
  ],
  "recommended_methodology": "Fine-tune a smaller vision-language model or utilize transfer learning with limited parameters due to CPU constraints and lack of labelled data.",
  "gaps": [
    {
      "gap": "Training without labelled data",
      "fix": "Use pre-trained labels or unsupervised learning methods.",
      "severity": "high"
    },
    {
      "gap": "Achieving 99% accuracy on unseen footage while measuring on training images",
      "fix": "Change the evaluation to use a separate validation set of unseen image data.",
      "severity": "hig

In [27]:
a3 = comp[(comp.test_id == "A3") & (comp.version == "repaired")].output.iloc[0]
rec = a3.split("RECOMMENDED METHODOLOGY:")[-1]

q = f"""Constraints: no labelled data and the user will not label any; laptop CPU only; one week; free tools only.
Recommendation: {rec}
Does this recommendation violate any constraint? Answer yes/no for each constraint, with one line why."""
print(call_model("", q))

1. **No labelled data**: Yes - The recommendation suggests obtaining a labelled dataset or creating synthetic labels, which violates this constraint.
2. **User will not label any**: Yes - The recommendation implies that the user would need to label data or create synthetic labels, which contradicts this constraint.
3. **Laptop CPU only**: No - The recommendation acknowledges the use of a smaller model that can run on a laptop's CPU, which complies with this constraint.
4. **One week**: No - The recommendation does not specify a time-consuming process, like labelling or complex model training, that exceeds one week, adhering to this constraint.
5. **Free tools only**: No - The recommendation does not mention any tools that would cost money, thus complying with this constraint.


The failure modes belong to the areas of 1) ambiguous instruction, 2) missing format, and 3) conflicting constraints.

One pattern I see relates to the fundamental architectural limitation of an LLM as a probabilistic system lacking ground-truth—it can produce persuasive and plausible methodological critiques while fundamentally misunderstanding the real-world research design that it is evaluating. Another pattern is a simple failure to follow prompt instructions, which even if locally rare could be unacceptably frequent at scale.

These patterns were most clearly demonstrated with the “conflicting constraints” class of adversarial input, which my best prompt struggled with even after repair. Specifically, (1) the model produced locally plausible statements without cross-checking them against each other for real-world feasibility, such as recommending data-labeling in clear constraint violation; and (2) did not double-check constraint pairs as prompted prior to output, such as a one-week completion timeframe.

This tells me that prompt engineering can mitigate but likely never eliminate fundamental LLM architectural limitations of a merely probabilistic rather than ground-truth understanding of reality. Prompts may also be partially ignored some of the time, which a schema may not catch if the output is formatted correctly. System-level solutions are likely required to mitigate these failure modes, such as a larger model, or a second verification pass.
